# **PARCIAL 1 TAM**

# **CRISTIAN DAVID CHALACA SALAS**

### 1.5 Implementación y Validación Numérica de Estimadores

Se implementaron y ajustaron los cuatro estimadores utilizando únicamente el conjunto de entrenamiento ($N_{\text{train}} = 240$, $N_{\text{test}} = 60$) generado a partir del espacio de características expandido $\Phi \in \mathbb{R}^{300 \times 20}$ ($Q = 20$). Se fijó el parámetro de regularización $\lambda = 1.0$ y el prior compatible $\sigma_w^2 = \sigma_\eta^2 / \lambda = 0.25$ con ruido $\sigma_\eta = 0.5$ ($\sigma_\eta^2 = 0.25$).

| Estimador | MSE Train | MSE Test | $\|\hat{w} - w_*\|_2$ |
| :--- | :---: | :---: | :---: |
| **Mínimos Cuadrados (OLS)** | $0.22085$ | $0.26961$ | $0.14877$ |
| **Máxima Verosimilitud (MLE)** | $0.22085$ | $0.26961$ | $0.14877$ |
| **Mínimos Cuadrados Reg. (Ridge)** | $0.22133$ | $0.27368$ | $0.15352$ |
| **Máximo a Posteriori (MAP)** | $0.22133$ | $0.27368$ | $0.15352$ |

**Análisis de Coincidencias y Discrepancias Numéricas:**

* **Coincidencia $\text{OLS} \equiv \text{MLE}$:** Se comprueba la equivalencia exacta de las soluciones ($\|\hat{w}_{\text{OLS}} - \hat{w}_{\text{MLE}}\|_2 = 0$). Maximizar la log-verosimilitud bajo ruido gaussiano i.i.d. equivale estrictamente a minimizar la suma de errores cuadráticos ordinarios.
* **Coincidencia $\text{Ridge} \equiv \text{MAP}$:** Al fijar el prior bayesiano como $\sigma_w^2 = \sigma_\eta^2 / \lambda$, la matriz de penalización MAP $(\Phi^\top\Phi + \frac{\sigma_\eta^2}{\sigma_w^2}I_Q)$ se reduce analíticamente a $(\Phi^\top\Phi + \lambda I_Q)$, arrojando estimaciones idénticas.
* **Precisión Numérica ($\epsilon$):** Las discrepancias de orden $\approx 10^{-16}$ reportadas en cómputo flotante corresponden únicamente a la precisión finita del estándar IEEE 754 durante la inversión de matrices en NumPy y no representan una desviación teórica.

In [ ]:
import numpy as np

# 1. FIJACIÓN DE SEMILLA Y GENERACIÓN DE DATOS SINTÉTICOS (X)
# Fijamos la semilla aleatoria para garantizar la reproducibilidad exacta
np.random.seed(42)

N = 300  # Número de muestras observadas
P = 5    # Dimensiones en el espacio original

# Generamos las filas de X de forma i.i.d con distribución N(0, 1)
X = np.random.normal(loc=0.0, scale=1.0, size=(N, P))
print(f"Dimensiones de la matriz de datos original X: {X.shape}")

# 2. CONSTRUCCIÓN DE LA MATRIZ DE DISEÑO EXPANDIDA (Phi)
# Se requiere mapear de P=5 características a Q=20 características:
# - Las 5 columnas originales.
# - Sus cuadrados (5 columnas).
# - Los productos cruzados por pares de columnas distintas: (5C2) = 10 columnas.
# Total = 5 + 5 + 10 = 20 características (Q=20).

columnas_originales = X
columnas_cuadrados = X ** 2

# Calculamos los productos cruzados por pares sin repetir combinaciones
productos_cruzados = []
for i in range(P):
    for j in range(i + 1, P):
        prod = X[:, i] * X[:, j] #evita multiplicar el mismo elemento 2 veces
        productos_cruzados.append(prod.reshape(-1, 1))

# Concatenamos horizontalmente las listas en una sola matriz
columnas_interaccion = np.hstack(productos_cruzados)

# Construimos la Matriz de Diseño final Phi combinando todos los bloques
Phi = np.hstack((columnas_originales, columnas_cuadrados, columnas_interaccion))
N, Q = Phi.shape
print(f"Dimensiones de la Matriz de Diseño Phi: {Phi.shape} (N={N}, Q={Q})")


# 3. GENERACIÓN DEL VECTOR DE RESPUESTAS (t) Y PARÁMETROS TEÓRICOS
# Definimos el ruido gaussiano eta_n ~ N(0, sigma_eta^2) con sigma_eta = 0.5
sigma_eta = 0.5
sigma_eta_sq = sigma_eta**2  # Varianza del ruido sigma_eta^2 = 0.25

eta = np.random.normal(loc=0.0, scale=sigma_eta, size=(N, 1))

# Generamos el vector latente real de pesos w_* ~ N(0, I_Q)
w_star = np.random.normal(loc=0.0, scale=1.0, size=(Q, 1))

# Construimos el vector objetivo siguiendo el modelo: t = Phi @ w_* + eta
t = Phi @ w_star + eta
print(f"Dimensiones del vector de respuestas t: {t.shape}")


# 4. PARTICIÓN DE LOS DATOS (ENTRENAMIENTO 80% / PRUEBA 20%)
N_train = int(0.8 * N)

# Dividimos secuencialmente las matrices y vectores de datos
Phi_train, Phi_test = Phi[:N_train, :], Phi[N_train:, :]
t_train, t_test = t[:N_train, :], t[N_train:, :]

print(f"\nDatos Entrenamiento (Train): Phi={Phi_train.shape}, t={t_train.shape}")
print(f"Datos Prueba (Test):         Phi={Phi_test.shape}, t={t_test.shape}")


# 5. AJUSTE DE LOS CUATRO ESTIMADORES ANALÍTICOS (ÚNICAMENTE CON TRAIN)
I_Q = np.identity(Q)

# Parámetros de regularización e hiperparámetros teóricos (Segunda convención LaTeX)
lam = 1.0                              # Parámetro de regularización lambda = 1.0
sigma_w_sq = sigma_eta_sq / lam        # Prior compatible sigma_w^2 = sigma_eta^2 / lambda = 0.25

# -[1.1] Mínimos Cuadrados Ordinarios (OLS)
# w_OLS = (Phi^T Phi)^(-1) Phi^T t
w_ols = np.linalg.inv(Phi_train.T @ Phi_train) @ Phi_train.T @ t_train

# -[1.2] Mínimos Cuadrados Regularizados (Ridge)
# w_REG = (Phi^T Phi + lambda * I_Q)^(-1) Phi^T t
w_ridge = np.linalg.inv(Phi_train.T @ Phi_train + lam * I_Q) @ Phi_train.T @ t_train

# -[1.3] Máxima Verosimilitud (MLE)
# w_MLE = (Phi^T Phi)^(-1) Phi^T t  (Coincidencia analítica con OLS)
w_mle = np.linalg.inv(Phi_train.T @ Phi_train) @ Phi_train.T @ t_train

# -[1.4] Máximo a Posteriori (MAP)
# w_MAP = (Phi^T Phi + (sigma_eta^2 / sigma_w^2) * I_Q)^(-1) Phi^T t
factor_bayesisano = sigma_eta_sq / sigma_w_sq
w_map = np.linalg.inv(Phi_train.T @ Phi_train + factor_bayesisano * I_Q) @ Phi_train.T @ t_train


# 6. EVALUACIÓN Y REPORTE DE ERRORES (MSE Y DISTANCIA ||w_est - w_*||_2)
def calcular_mse(Phi_matrix, t_vector, w_vector):
    predicciones = Phi_matrix @ w_vector
    return np.mean((t_vector - predicciones) ** 2)

estimadores = {
    "Mínimos Cuadrados (OLS)": w_ols,
    "Mínimos Cuadrados Reg. (Ridge)": w_ridge,
    "Máxima Verosimilitud (MLE)": w_mle,
    "Máximo a Posteriori (MAP)": w_map
}

print(f"\n{'Estimador':<32} | {'MSE Train':<10} | {'MSE Test':<10} | {'||w - w_*||_2':<12}")
print("-" * 75)

for nombre, w_est in estimadores.items():
    mse_train = calcular_mse(Phi_train, t_train, w_est)
    mse_test = calcular_mse(Phi_test, t_test, w_est)
    # Distancia Euclidiana (norma L2) respecto al vector verdadero w_*
    distancia_w = np.linalg.norm(w_est - w_star)

    print(f"{nombre:<32} | {mse_train:<10.5f} | {mse_test:<10.5f} | {distancia_w:<12.5f}")


# 7. VERIFICACIÓN RIGUROSA DE COINCIDENCIAS Y MANEJO DE ÉPSILON
# Tolerancia absoluta de precisión numérica de coma flotante (epsilon = 1e-12)
eps_tol = 1e-12

diferencia_ols_mle = np.linalg.norm(w_ols - w_mle)
diferencia_ridge_map = np.linalg.norm(w_ridge - w_map)

print("\n--- Justificaciones para la Sustentación Oral ---")
print(f"¿OLS y MLE son exactamente iguales?: {diferencia_ols_mle < eps_tol} (Diferencia norma: {diferencia_ols_mle:.2e})")
print(f"¿Ridge y MAP son exactamente iguales?: {diferencia_ridge_map < eps_tol} (Diferencia norma: {diferencia_ridge_map:.2e})")

print("\nExplicación teórica de discrepancias de coma flotante (Épsilon):")
print(f"Las diferencias entre estimadores teóricamente idénticos son de orden {diferencia_ols_mle:.2e}.")
print("Estas pequeñas discrepancias NO son un error teórico, sino un épsilon de precisión")
print("de coma flotante (IEEE 754) introducido por las operaciones de inversión matricial de NumPy.")

Dimensiones de la matriz de datos original X: (300, 5)
Dimensiones de la Matriz de Diseño Phi: (300, 20) (N=300, Q=20)
Dimensiones del vector de respuestas t: (300, 1)

Datos Entrenamiento (Train): Phi=(240, 20), t=(240, 1)
Datos Prueba (Test):         Phi=(60, 20), t=(60, 1)

Estimador                        | MSE Train  | MSE Test   | ||w - w_*||_2
---------------------------------------------------------------------------
Mínimos Cuadrados (OLS)          | 0.22085    | 0.26961    | 0.14877     
Mínimos Cuadrados Reg. (Ridge)   | 0.22133    | 0.27368    | 0.15352     
Máxima Verosimilitud (MLE)       | 0.22085    | 0.26961    | 0.14877     
Máximo a Posteriori (MAP)        | 0.22133    | 0.27368    | 0.15352     

--- Justificaciones para la Sustentación Oral ---
¿OLS y MLE son exactamente iguales?: True (Diferencia norma: 0.00e+00)
¿Ridge y MAP son exactamente iguales?: True (Diferencia norma: 0.00e+00)

Explicación teórica de discrepancias de coma flotante (Épsilon):
Las diferencia

### 2.4 Comparación Experimental MMD entre MNIST y Fashion-MNIST

Para comparar las distribuciones $P_{\text{MNIST}}$ y $P_{\text{Fashion}}$, se tomaron muestras independientes sin etiquetas de $n = m = 500$ imágenes ($P = 784$ componentes en $[0, 1]$). Se empleó un kernel gaussiano $k(u,v) = \exp\left(-\frac{\|u-v\|_2^2}{2\sigma^2}\right)$, determinando el ancho de banda mediante la *Heurística de la Mediana* sobre la muestra combinada, obteniendo $\sigma = 11.2255$.

Las dimensiones de las matrices de Gram resultantes para cada evaluación son $K_{XX} \in \mathbb{R}^{500 \times 500}$, $K_{YY} \in \mathbb{R}^{500 \times 500}$ y $K_{XY} \in \mathbb{R}^{500 \times 500}$, evaluándose la forma matricial con vectores columna de unos $1_{500} \in \mathbb{R}^{500 \times 1}$:

$$\hat{\text{MMD}}_b^2 = \frac{1}{n^2} 1_n^\top K_{XX} 1_n + \frac{1}{m^2} 1_m^\top K_{YY} 1_m - \frac{2}{nm} 1_n^\top K_{XY} 1_m$$

| Comparación de Distribuciones | $\hat{\text{MMD}}_b^2$ |
| :--- | :---: |
| **MNIST (G1) vs. Fashion-MNIST (G1)** | $0.131625$ |
| **MNIST (G1) vs. MNIST (G2 - Disjunto)** | $0.001640$ |
| **Fashion-MNIST (G1) vs. Fashion-MNIST (G2 - Disjunto)** | $0.001077$ |

**Conclusiones y Aspectos Metodológicos:**

1. **Separabilidad Topológica:** La discrepancia entre conjuntos de distinta naturaleza (dígitos vs. ropa) es casi dos órdenes de magnitud mayor ($\approx 0.1316$) que las comparaciones intra-dataset ($\approx 0.001$), demostrando que el embedding de media en el RKHS discrimina las distribuciones de forma inyectiva.
2. **Sesgo Muestral:** Las pruebas intra-dataset no arrojan un cero absoluto debido a las fluctuaciones de muestreo finito ($n=500$) y al sesgo positivo del estimador $\mathcal{O}(1/n)$ al conservar los elementos autorreferenciales de la diagonal ($k(x_i, x_i) = 1$).
3. **Contraste de Hipótesis:** Un valor absoluto de MMD resulta insuficiente para emitir un juicio estadístico definitivo. Se requiere un marco formal de prueba de hipótesis ($H_0: P_X = P_Y$) mediante pruebas de permutación (*permutation tests*) para estimar la distribución nula empírica y calcular un p-valor confiable.

In [ ]:
import numpy as np
import tensorflow as tf
from scipy.spatial.distance import pdist

# 1. CARGA Y MUESTREO REPRODUCIBLE (n = m = 500, P = 784)
np.random.seed(42)

# Cargar MNIST y Fashion-MNIST normalizados a [0, 1]
(X_mnist_all, _), _ = tf.keras.datasets.mnist.load_data()
X_mnist_flat = X_mnist_all.reshape(-1, 28 * 28) / 255.0

(X_fashion_all, _), _ = tf.keras.datasets.fashion_mnist.load_data()
X_fashion_flat = X_fashion_all.reshape(-1, 28 * 28) / 255.0

# 1000 imágenes aleatorias por dataset
idx_m = np.random.choice(X_mnist_flat.shape[0], 1000, replace=False)
idx_f = np.random.choice(X_fashion_flat.shape[0], 1000, replace=False)

# Partición en dos grupos disjuntos de 500 muestras (n = m = 500)
mnist_g1, mnist_g2 = X_mnist_flat[idx_m[:500]], X_mnist_flat[idx_m[500:]]
fashion_g1, fashion_g2 = X_fashion_flat[idx_f[:500]], X_fashion_flat[idx_f[500:]]

print(f"Muestras preparadas: n = m = 500 vectores de dimensión {mnist_g1.shape[1]}")


# 2. HEURÍSTICA DE LA MEDIANA PARA ANCHO DE BANDA SIGMA
muestra_combinada = np.vstack((mnist_g1, fashion_g1))
distancias_pares = pdist(muestra_combinada, metric='euclidean')
sigma_optimo = np.median(distancias_pares)

print(f"Ancho de banda sigma (Heurística de la Mediana): {sigma_optimo:.4f}")


# 3. KERNEL GAUSSIANO Y ESTIMADOR EMPÍRICO MATRICIAL DE MMD_b^2
def calcular_kernel_gaussiano(X_mat, Y_mat, sigma):
    norm_X = np.sum(X_mat**2, axis=1).reshape(-1, 1)
    norm_Y = np.sum(Y_mat**2, axis=1).reshape(-1, 1)
    # ||u - v||^2 = ||u||^2 + ||v||^2 - 2<u, v>
    dist_cuadrada = norm_X @ np.ones((1, Y_mat.shape[0])) + np.ones((X_mat.shape[0], 1)) @ norm_Y.T - 2 * (X_mat @ Y_mat.T)
    return np.exp(-dist_cuadrada / (2 * (sigma**2)))

def calcular_mmd2_sesgado_matricial(X, Y, sigma):
    n = X.shape[0]
    m = Y.shape[0]

    # Matrices de Gram
    K_XX = calcular_kernel_gaussiano(X, X, sigma)  # (n x n)
    K_YY = calcular_kernel_gaussiano(Y, Y, sigma)  # (m x m)
    K_XY = calcular_kernel_gaussiano(X, Y, sigma)  # (n x m)

    # Vectores columna de unos 1_n y 1_m
    ones_n = np.ones((n, 1))
    ones_m = np.ones((m, 1))

    # Formulación cuadrática exacta: (1/n^2) 1_n^T K_XX 1_n + (1/m^2) 1_m^T K_YY 1_m - (2/nm) 1_n^T K_XY 1_m
    term_XX = (ones_n.T @ K_XX @ ones_n) / (n**2)
    term_YY = (ones_m.T @ K_YY @ ones_m) / (m**2)
    term_XY = (ones_n.T @ K_XY @ ones_m) / (n * m)

    MMD2_b = term_XX + term_YY - 2 * term_XY
    return MMD2_b.item()


# 4. CÓMPUTO Y REPORTES
mmd2_inter = calcular_mmd2_sesgado_matricial(mnist_g1, fashion_g1, sigma_optimo)
mmd2_intra_mnist = calcular_mmd2_sesgado_matricial(mnist_g1, mnist_g2, sigma_optimo)
mmd2_intra_fashion = calcular_mmd2_sesgado_matricial(fashion_g1, fashion_g2, sigma_optimo)

print("\n" + "="*65)
print(f"{'Comparación Estadística':<42} | {'MMD_b^2 Estimada':<15}")
print("="*65)
print(f"{'MNIST (G1) vs Fashion-MNIST (G1)':<42} | {mmd2_inter:<15.6f}")
print(f"{'MNIST (G1) vs MNIST (G2 - Disjunto)':<42} | {mmd2_intra_mnist:<15.6f}")
print(f"{'Fashion-MNIST (G1) vs Fashion (G2)':<42} | {mmd2_intra_fashion:<15.6f}")
print("="*65)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Muestras preparadas: n = m = 500 vectores de dimensión 784
Ancho de banda sigma (Heurística de la Mediana): 11.2255

Comparación Estadística                    | MMD_b^2 Estimada
MNIST (G1) vs Fashion-MNIST (G1)           | 0.131625       
MNIST (G1) vs MNIST (G2 - Disjunto)        | 0.001640       
Fashion-MNIST (G1) vs Fashion (G2)         | 0.001077       
